Week 15 · Day 5 — Evaluating Retrieval: Precision@k & Recall@k
Why this matters

A RAG system is only as good as its retriever. If the right documents aren’t retrieved, the LLM has no chance to answer correctly. We need metrics like precision@k and recall@k to measure retrieval quality, just like accuracy in classification.

Theory Essentials

Ground truth: which docs are truly relevant to a query.

Precision@k: fraction of retrieved docs (top-k) that are relevant.

Recall@k: fraction of all relevant docs that appear in top-k.

High precision = fewer wrong docs retrieved.

High recall = fewer correct docs missed.

In RAG, both matter: recall ensures relevant info is present; precision ensures less noise.

In [1]:
# Setup
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Docs
docs = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Prado Museum is in Madrid.",
    "The Brandenburg Gate is in Berlin.",
    "Big Ben is in London.",
    "The Acropolis is in Athens."
]

# Embeddings + FAISS
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs).astype("float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# Retrieval
def retrieve(query, k=3):
    q_emb = model.encode([query]).astype("float32")
    D, I = index.search(q_emb, k)
    return [docs[i] for i in I[0]]

# Ground truth labels
ground_truth = {
    "Where is the Colosseum?": ["The Colosseum is in Rome."],
    "Where is Big Ben located?": ["Big Ben is in London."],
    "Which monuments are in Berlin?": ["The Brandenburg Gate is in Berlin."],
}

# Metrics
def precision_recall(query, k=3):
    retrieved = retrieve(query, k)
    relevant = ground_truth[query]
    true_pos = sum(1 for d in retrieved if d in relevant)
    precision = true_pos / k
    recall = true_pos / len(relevant)
    return precision, recall

# Example
for q in ground_truth:
    p, r = precision_recall(q, k=3)
    print(f"{q}\n Precision@3={p:.2f}, Recall@3={r:.2f}\n")


Where is the Colosseum?
 Precision@3=0.33, Recall@3=1.00

Where is Big Ben located?
 Precision@3=0.33, Recall@3=1.00

Which monuments are in Berlin?
 Precision@3=0.33, Recall@3=1.00



1) Core (10–15 min)

Task: Run the metric on the query “Where is Big Ben located?” and check precision@1 and recall@1.

In [2]:
p, r = precision_recall("Where is Big Ben located?", k=1)
print("Precision@1:", p, "Recall@1:", r)


Precision@1: 1.0 Recall@1: 1.0


2) Practice (10–15 min)

Task: Add a new query: “Which landmarks are in Paris?” with ground truth = [“The Eiffel Tower is in Paris.”]. Evaluate precision@2 and recall@2.

In [3]:
ground_truth["Which landmarks are in Paris?"] = ["The Eiffel Tower is in Paris."]
p, r = precision_recall("Which landmarks are in Paris?", k=2)
print("Precision@2:", p, "Recall@2:", r)


Precision@2: 0.5 Recall@2: 1.0


3) Stretch (optional, 10–15 min)

Task: Modify precision_recall to also return the retrieved docs, so you can debug false positives.

In [4]:
def precision_recall_debug(query, k=3):
    retrieved = retrieve(query, k)
    relevant = ground_truth[query]
    true_pos = sum(1 for d in retrieved if d in relevant)
    precision = true_pos / k
    recall = true_pos / len(relevant)
    return precision, recall, retrieved

print(precision_recall_debug("Where is the Colosseum?", 3))


(0.3333333333333333, 1.0, ['The Colosseum is in Rome.', 'The Acropolis is in Athens.', 'The Prado Museum is in Madrid.'])


Mini-Challenge (≤40 min)

Evaluate Your Retriever

Add at least 3 new queries + ground truth answers.

Compute precision@k and recall@k for k=1, 2, 3.

Write a small table of results (query vs P@k, R@k).

Acceptance Criteria:

At least 5 queries total in ground_truth.

Metrics reported consistently for multiple values of k.

You can spot at least 1 case where recall < 1.0.

In [5]:
# --- Add ≥3 new queries (make some multi-label to get recall<1) ---
ground_truth.update({
    "Where is the Eiffel Tower?": ["The Eiffel Tower is in Paris."],
    "Which monuments are in Paris or Berlin?": [
        "The Eiffel Tower is in Paris.",
        "The Brandenburg Gate is in Berlin."
    ],
    "Which monuments are in Rome or Athens?": [
        "The Colosseum is in Rome.",
        "The Acropolis is in Athens."
    ],
})

# --- Metrics for multiple k values ---
def precision_recall_at_k(query: str, k: int):
    retrieved = retrieve(query, k)
    relevant = set(ground_truth[query])
    tp = sum(1 for d in retrieved if d in relevant)
    precision = tp / k
    recall = tp / len(relevant)
    return precision, recall

def evaluate(ks=(1,2,3)):
    # header
    header = ["Query"] + [f"P@{k}" for k in ks] + [f"R@{k}" for k in ks]
    widths = [50] + [6]* (2*len(ks))
    print(" | ".join(h.ljust(w) for h,w in zip(header,widths)))
    print("-"* (sum(widths) + 3*(len(widths)-1)))

    for q in ground_truth:
        pvals, rvals = [], []
        for k in ks:
            p, r = precision_recall_at_k(q, k)
            pvals.append(f"{p:.2f}")
            rvals.append(f"{r:.2f}")
        row = [q[:50].ljust(50)] + [v.ljust(6) for v in (pvals + rvals)]
        print(" | ".join(row))

# --- Run evaluation ---
evaluate(ks=(1,2,3))


Query                                              | P@1    | P@2    | P@3    | R@1    | R@2    | R@3   
--------------------------------------------------------------------------------------------------------
Where is the Colosseum?                            | 1.00   | 0.50   | 0.33   | 1.00   | 1.00   | 1.00  
Where is Big Ben located?                          | 1.00   | 0.50   | 0.33   | 1.00   | 1.00   | 1.00  
Which monuments are in Berlin?                     | 1.00   | 0.50   | 0.33   | 1.00   | 1.00   | 1.00  
Which landmarks are in Paris?                      | 1.00   | 0.50   | 0.33   | 1.00   | 1.00   | 1.00  
Where is the Eiffel Tower?                         | 1.00   | 0.50   | 0.33   | 1.00   | 1.00   | 1.00  
Which monuments are in Paris or Berlin?            | 1.00   | 1.00   | 0.67   | 0.50   | 1.00   | 1.00  
Which monuments are in Rome or Athens?             | 1.00   | 1.00   | 0.67   | 0.50   | 1.00   | 1.00  


Notes / Key Takeaways

Retrieval evaluation is as important as generation.

Precision@k and Recall@k give quantitative feedback.

Good retrieval = high recall (correct docs included) + high precision (few wrong docs).

If retrieval fails, the LLM can’t fix it.

Evaluating retrievers is key for improving RAG systems.

Reflection

Why might you prioritize recall over precision in some RAG use cases?

If recall@k = 1.0 but precision is low, what does that mean in practice?



**Why might you prioritize recall over precision in some RAG use cases?**

* In QA or research settings, it’s often better to **retrieve all relevant info**, even if some irrelevant docs are included.
* The LLM can then filter or summarize, but if the retriever misses key docs (low recall), the model will never see the correct info.

**If recall@k = 1.0 but precision is low, what does that mean in practice?**

* It means the retriever successfully included the correct document(s), but also pulled in a lot of **irrelevant noise**.
* The answer can still be correct (since the right doc is present), but the LLM has to work harder to sift through extra context, which may increase hallucinations or irrelevant details.


